# MaxEnt FRET distance MEM: Python API example

This notebook shows how to call the MaxEnt FRET distance solver using the
convenience functions in `api.py`. For simplicity we use synthetic data;
the focus is on how to call the API and inspect the recovered distance
distribution `p(R)`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from chisurf.plugins.fluorescence_decay.maxent_decay.fmem.api import (
    run_fret_mem_from_arrays,
    build_distance_grid,
)

# --- 1. Build a simple synthetic FRET-like decay + IRF ---
dt = 0.02  # ns per channel
n_channels = 512
t = np.arange(n_channels) * dt

# Toy IRF: Gaussian peak at 1 ns
irf = np.exp(-0.5 * ((t - 1.0) / 0.1) ** 2)
irf /= irf.sum()

# Toy decay: mixture of two exponentials mimicking FRET and donor-only
tau_D = 4.0   # donor-only lifetime [ns]
tau_F = 1.0   # FRET-shortened lifetime [ns]
frac_F = 0.6  # FRET fraction
decay_do = np.exp(-t / tau_D)
decay_fret = np.exp(-t / tau_F)
decay_ideal = frac_F * decay_fret + (1.0 - frac_F) * decay_do

decay_conv = np.convolve(decay_ideal, irf, mode="full")[:n_channels]

# Add Poisson noise
rng = np.random.default_rng(1)
scale = 2e4
decay_counts = rng.poisson(decay_conv * scale).astype(float)
irf_counts = irf * decay_counts.max() * 0.5

# --- 2. Run the MaxEnt FRET solver via the API ---
R_grid = build_distance_grid(r_min=18.0, r_max=80.0, r_step=0.5)

result = run_fret_mem_from_arrays(
    decay=decay_counts,
    irf=irf_counts,
    dt=dt,
    R=R_grid,
    tau0=tau_D,
    R0=52.0,
    x_donly=1.0 - frac_F,
    lamp_scatter=0.0,
    fit_start_fraction=0.9,
    nu=5e-2,
    use_periodic=False,
)

# --- 3. Inspect the result ---
R = np.asarray(result["R"], dtype=float).ravel()
p = np.asarray(result["p"], dtype=float).ravel()
p /= p.sum() if p.sum() > 0 else 1.0

Fi = np.asarray(result["Fi"], dtype=float)
y_seg = np.asarray(result["y"], dtype=float).ravel()
sigma = np.asarray(result["sigma"], dtype=float).ravel()
fitstart, fitstop = result["fitrange"]

fit_seg = (Fi @ p) * sigma
fit_full = np.zeros_like(decay_counts)
fit_full[fitstart : fitstop + 1] = fit_seg

# --- 4. Plot decay + fit and distance distribution ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].semilogy(t, np.maximum(decay_counts, 1.0), label="data")
axes[0].semilogy(t, np.maximum(fit_full, 1.0), label="MaxEnt fit")
axes[0].set_xlabel("time [ns]")
axes[0].set_ylabel("counts")
axes[0].legend()

axes[1].plot(R, p, marker="o")
axes[1].set_xlabel("distance R [Å]")
axes[1].set_ylabel("probability")
axes[1].set_title("Distance distribution p(R)")

fig.tight_layout()
fig.show()
